In [ ]:
# Parameters (overridden by the RunNotebook job)
src_server = ""      # serverFqdn of source_db (from GET /sqlDatabases)
src_db = ""          # databaseName of source_db
ep_server = ""       # SQL analytics endpoint host (GET /sqlEndpoints/{id}/connectionString)
ep_db = "source_db"
seed = 42
job = "workload"
rates = "10,100,1000"   # rows/s, run order is interleaved per repeat
repeats = 3
run_s = 150             # burst length per run
gap_s = 60              # idle between runs
marker_every_s = 10
poll_s = 2
tail_timeout_s = 600
idle_s = 600            # cold probes only
n_probes = 3            # cold probes only


In [ ]:
import json, struct, time, datetime, random, os, traceback, statistics
import pyodbc, notebookutils
OUT = f"/lakehouse/default/Files/mirror_lag/{job}"
os.makedirs(OUT, exist_ok=True)
def utcnow(): return datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)
def conn(server, db):
    t = notebookutils.credentials.getToken("https://database.windows.net/")
    b = t.encode("utf-16-le"); ts = struct.pack(f"<I{len(b)}s", len(b), b)
    return pyodbc.connect(f"Driver={{ODBC Driver 18 for SQL Server}};Server={server};Database={db};Encrypt=yes;",
                          attrs_before={1256: ts}, autocommit=True)
def save(name, obj):
    with open(f"{OUT}/{name}", "w") as f:
        json.dump(obj, f, indent=1, default=str)
def offset_src(c, n=5):
    """seconds to add to the source clock to get notebook UTC (min-RTT sample)"""
    best = None
    for _ in range(n):
        t0 = time.time(); s = c.execute("SELECT SYSUTCDATETIME()").fetchone()[0]; t1 = time.time()
        rtt = t1 - t0; off = (t0 + t1) / 2 - s.replace(tzinfo=datetime.timezone.utc).timestamp()
        if best is None or rtt < best[0]: best = (rtt, off)
    return {"rtt_s": round(best[0], 4), "offset_s": round(best[1], 4)}
MARK_BASE = 9_000_000_000
def put_marker(src, mid, tag):
    r = src.execute("SET NOCOUNT ON; INSERT INTO dbo.table_a(id, segment, attribute_a, value, updated_at) "
                    "OUTPUT inserted.updated_at VALUES (?, 'M', ?, 0, SYSUTCDATETIME());", mid, tag).fetchone()
    return r[0]
def replication_dmvs(src):
    res = {}
    for q in ["SELECT TOP 30 * FROM sys.dm_change_feed_log_scan_sessions ORDER BY start_time DESC",
              "SELECT TOP 30 * FROM sys.dm_change_feed_errors ORDER BY entry_time DESC"]:
        try:
            cur = src.execute(q); cols = [d[0] for d in cur.description]
            res[q] = [dict(zip(cols, [str(v) for v in row])) for row in cur.fetchall()]
        except Exception as e:
            res[q] = "ERROR: " + str(e)
    return res


In [ ]:
log = {"job": job, "seed": seed}
src = conn(src_server, src_db); ep = conn(ep_server, ep_db)
log["offset"] = offset_src(src)
src.execute("""IF OBJECT_ID('dbo.table_a') IS NULL
CREATE TABLE dbo.table_a (
  id bigint NOT NULL CONSTRAINT pk_table_a PRIMARY KEY CLUSTERED,
  segment char(1) NOT NULL,
  attribute_a varchar(12) NOT NULL,
  value decimal(18,2) NOT NULL,
  updated_at datetime2(6) NOT NULL)""")
n_have = src.execute("SELECT COUNT_BIG(*) FROM dbo.table_a").fetchone()[0]
log["rows_before"] = n_have
t0 = utcnow(); log["seed_start_utc"] = t0
# deterministic seed rows: CHECKSUM of (id, seed) drives every column, so reruns give identical data
for lo in range(1, 1_000_001, 250_000):
    src.execute(f"""SET NOCOUNT ON;
INSERT INTO dbo.table_a(id, segment, attribute_a, value, updated_at)
SELECT g.value,
  CHAR(65 + ABS(CHECKSUM(g.value, {seed}, 1)) % 3),
  CONCAT('attr_', RIGHT(CONCAT('000', ABS(CHECKSUM(g.value, {seed}, 2)) % 1000), 4)),
  CAST(ABS(CHECKSUM(g.value, {seed}, 3)) % 100000 AS decimal(18,2)) / 100,
  DATEADD(second, ABS(CHECKSUM(g.value, {seed}, 4)) % 86400, CAST('2026-01-01' AS datetime2(6)))
FROM GENERATE_SERIES(CAST({lo} AS bigint), CAST({lo + 249_999} AS bigint)) g
WHERE NOT EXISTS (SELECT 1 FROM dbo.table_a t WHERE t.id = g.value);""")
t1 = utcnow(); log["seed_end_utc"] = t1; log["seed_s"] = (t1 - t0).total_seconds()
src_n = src.execute("SELECT COUNT_BIG(*) FROM dbo.table_a").fetchone()[0]; log["src_rows"] = src_n
# poll the SQL analytics endpoint until the full seed is visible
polls = []; seen = None; deadline = time.time() + 1800
while time.time() < deadline:
    ps = utcnow()
    try:
        n = ep.execute(f"SELECT COUNT_BIG(*) FROM dbo.table_a WHERE {len(polls)} = {len(polls)}").fetchone()[0]
        err = None
    except Exception as e:
        n, err = None, str(e)[:300]
    polls.append({"t": ps, "rows": n, "err": err})
    if n == src_n: seen = ps; break
    time.sleep(5)
log["first_full_visible_utc"] = seen
log["initial_snapshot_lag_s"] = (seen - t1).total_seconds() if seen else None
log["polls"] = polls[-20:]; log["n_polls"] = len(polls)
log["dmvs"] = replication_dmvs(src)
save("seed.json", log)
print(json.dumps({k: v for k, v in log.items() if k not in ("dmvs", "polls")}, default=str, indent=1))
